## Churn Analysis and Customer Intelligence


In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as pyplot
import seaborn as sns
import sqlite3 

## 1. Import database / data  

In [3]:
# NOTE:
# If you encounter any issues importing the customer_churn.db file,
# run the code below to create the database locally from the provided Excel file OR, can directly import data & start working.

# Otherwise, skip this step and proceed to the next section: Data Import

# Excel file path
# excel_file = 'customer_churn_data_raw.xlsx'

excel_file = 'customer_churn_data_raw.xlsx'

# Create SQLite database connection
# conn = sqlite3.connect('customer_churn.db')

conn  = sqlite3.connect('customer_churn.db')

# Read all sheet names from Excel file
# excel_data = pd.ExcelFile(excel_file)

excel_data = pd.ExcelFile(excel_file)

# Loop through each sheet and store as separate table
# for sheet in excel_data.sheet_names:
#     df = pd.read_excel(excel_file, sheet_name=sheet) # Read sheet into dataframe
#     # Write dataframe to SQLite table
#     df.to_sql(
#         name=sheet,          # Table name = Sheet name
#         con=conn,
#         if_exists='replace', # Replace table if already exists
#         index=False
#     )


for sheet in excel_data.sheet_names:
    df = pd.read_excel(excel_file , sheet_name = sheet)  # read sheet into dataframe
     # Write dataframe to SQLite table
    df.to_sql(
        name=sheet,
        con=conn,
        if_exists = 'replace',  # Replace table if already exists
        index=False
    )
    


# Close connection
# conn.close()

conn.close()

print("All sheets successfully converted into SQLite tables.")

All sheets successfully converted into SQLite tables.


In [6]:
# Data Import: db file to pandas, storing each table to a separate df

# Connect to SQLite database

conn = sqlite3.connect('customer_churn.db')

# sql query to Get all table names

sql_query = """SELECT name 
            FROM sqlite_master
            WHERE type='table';
            """

# read sql query in pandas

tables = pd.read_sql_query(sql_query, conn)


# create dataframe for each table

for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)  # Read table into dataframe
    globals()[f"df_{table_name}"] = df   # Create dynamic dataframe name
    print(f"Created dataframe: df_{table_name}")
    



# Close connection

conn.close()









Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [7]:
# Print table names and column names

conn = sqlite3.connect('customer_churn.db')

In [8]:
for table_name in tables['name']:
    print(f"\n Table Name : {table_name}")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql_query(columns_query, conn)
    print("Columns:")

    print(columns['name'].tolist())


# Close connection
conn.close()


 Table Name : db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

 Table Name : db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

 Table Name : db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


In [ ]:
# PRAGMA is a special command in SQLite used to:
# inspect db information, control db settings, retrieve metadata about tables

## 2. Data cleaning

In [ ]:
df_db_customer.head() # customer table 
df_db_customer.tail() # customer table 

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,None,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,None,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,None,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,None,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,None,None


In [11]:
df_db_customer.info() # customer table

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     object
 1   name        21 non-null     object
 2   country     18 non-null     object
 3   state       21 non-null     object
 4   gender      21 non-null     object
 5   dob         21 non-null     object
 6   interests   4 non-null      object
 7   pincode     0 non-null      object
dtypes: object(8)
memory usage: 1.4+ KB


In [ ]:
# a. rename col - name to customer_name
# b. drop columns - interest and pincode
# c. change data type - dob
# d. data standardization - gender
# e. fix missing values (using existing data) - country

#### a. Rename Col

In [ ]:
# a. rename col - name to customer_name

df_db_customer.rename(columns = {'name' : 'customer_name'}, inplace= True)

#### b. Drop columns
